# Lab 2 — Feed-forward nets and Ch. 2 language-model tools

DATA 422 · ungraded · *Hundred-Page Language Models Book* Ch. 2

Colab, **CPU**. We start with a **toy regression** ($\sin x$ + noise) so you can see nonlinear fitting with `nn.Module`, then Ch. 2 ideas (tokenization, softmax, perplexity). Wildfire Model 1 is on the [Project 1 stepping stone](../stepping-stones/project-1-stone-01.qmd), not in this lab.

Friday **paper** quiz: [Quiz 2 study bank](https://roverhol.github.io/data422/quizzes/quiz-02-prep.html).


## Learning objectives

By the end of this lab you can:

1. Write a feed-forward net as `class ... (nn.Module)` and match **equation ↔ diagram ↔ code**.
2. Fit a 1-D nonlinear target ($\sin x$ + noise), plot data / truth / prediction, and change depth, width, or activation from a config block.
3. Switch the same skeleton from **numeric** output (MSE) to **binary** (one logit + `BCEWithLogitsLoss`) or **three-class** (three logits + `cross_entropy`).
4. Tokenize short text the way Ch. 2 / [theLMbook notebooks](https://github.com/aburkov/theLMbook) do, look up `nn.Embedding`, and compute **perplexity** from average NLL.


In [ ]:
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(422)
np.random.seed(422)
device = torch.device("cpu")


## Part 0 — JIT: object-oriented Python (90 seconds)

PyTorch models are **objects**: grouped data + methods. Not a full OOP course — just enough to read `class SinMLP(nn.Module)` below.

| Idea | Meaning |
|------|---------|
| `class Dog:` | blueprint for one kind of thing |
| `def __init__(self, name):` | **constructor** — runs when you write `Dog("Ada")` |
| `self` | *this* particular object (like `this` in Java, `self` in Python) |
| `self.name = name` | store on the object so other methods can use it |
| `def bark(self):` | **method** — first argument is always `self` |

**Training rule:** put layers on `self` (`self.lin = nn.Linear(...)`). A bare `lin = nn.Linear(...)` inside `__init__` is **not** registered and will **not** train.


In [ ]:
class Counter:
    def __init__(self):
        self.n = 0

    def bump(self):
        self.n += 1

c = Counter()
c.bump()
print("n =", c.n)
# Desired: n = 1


### `nn.Module` — same pattern, plus autograd

Lab 1 used `nn.Sequential` you did not author. Lab 2 writes **subclasses** of `nn.Module`:

| Piece | Role |
|-------|------|
| `class Foo(nn.Module)` | `Foo` is a Module — gets `.parameters()`, `train()` / `eval()` |
| `super().__init__()` | call parent constructor **first** |
| `self.lin = nn.Linear(...)` | store submodules on `self` |
| `def forward(self, x)` | the math; return **logits** (or raw $\hat y$ for regression) |
| `model(x)` | calls `forward`; use this in training (not `model.forward(x)`) |

You do **not** write `backward`.


In [ ]:
class LinearLogit(nn.Module):
    def __init__(self, n_in: int):
        super().__init__()
        self.lin = nn.Linear(n_in, 1)

    def forward(self, x):
        return self.lin(x).squeeze(-1)

m = LinearLogit(3)
x = torch.randn(4, 3)
print("out", tuple(m(x).shape), "params", sum(p.numel() for p in m.parameters()))
# Desired: out (4,)  params 4

# JIT check: layers must live on self
class Broken(nn.Module):
    def __init__(self):
        super().__init__()
        lin = nn.Linear(2, 1)  # not self.lin — will NOT train

print("broken params", sum(p.numel() for p in Broken().parameters()))
# Desired: broken params 0


## Part 1 — A configurable MLP for $\sin x$ + noise

### Notation (book style)

One scalar input $x \in \mathbb{R}$ (so $\mathbf{x} \in \mathbb{R}^{1}$). Hidden layers use activation $\sigma$ (ReLU, tanh, …). For $L$ weight layers:

$$
\mathbf{h}^{(0)} = \mathbf{x}, \qquad
\mathbf{h}^{(\ell)} = \sigma\!\left(\mathbf{W}^{(\ell)} \mathbf{h}^{(\ell-1)} + \mathbf{b}^{(\ell)}\right), \quad \ell = 1,\ldots,L-1.
$$

**Regression head** (numeric $\hat y$):

$$
\hat y = \mathbf{W}^{(L)} \mathbf{h}^{(L-1)} + b^{(L)}.
$$

Match the diagram:

```
x ──► [Linear + σ] ──► [Linear + σ] ──► … ──► [Linear] ──► ŷ
       W⁽¹⁾,b⁽¹⁾         W⁽²⁾,b⁽²⁾              W⁽ᴸ⁾,b⁽ᴸ⁾
```

| Math | PyTorch in `forward` |
|------|----------------------|
| $\mathbf{h}^{(\ell)} = \sigma(\mathbf{W}^{(\ell)}\mathbf{h}^{(\ell-1)}+\mathbf{b}^{(\ell)})$ | `h = self.act(self.layers[ℓ](h))` |
| $\hat y = \mathbf{W}^{(L)}\mathbf{h}^{(L-1)}+b^{(L)}$ | `return self.out(h).squeeze(-1)` |
| mean squared error | `nn.MSELoss()` on $\hat y$ vs $y$ |

Change **`HIDDEN_SIZES`**, **`ACTIVATION`**, and **`N_EPOCHS`** in the next cell — the rest of Part 1 follows.


In [ ]:
# ── knobs you will change in exercises ──
HIDDEN_SIZES = [32, 32]   # list of hidden widths; try [8], [64, 64, 32], …
ACTIVATION = nn.ReLU      # try nn.Tanh, nn.GELU, …
N_EPOCHS = 800
LR = 1e-2
NOISE_STD = 0.15

n = 400
x_np = np.linspace(-10, 10, n, dtype=np.float32)
y_true_np = np.sin(x_np)
y_np = y_true_np + np.random.normal(0, NOISE_STD, size=n).astype(np.float32)

x = torch.from_numpy(x_np).unsqueeze(1)   # (n, 1)
y = torch.from_numpy(y_np)                # (n,)


In [ ]:
class SinMLP(nn.Module):
    """Fully connected net: R^{n_in} -> R^{n_out}. Hidden layers use ACTIVATION."""

    def __init__(self, n_in: int, hidden_sizes: list[int], n_out: int, activation: type[nn.Module] = nn.ReLU):
        super().__init__()
        self.act = activation()
        layers: list[nn.Module] = []
        prev = n_in
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            prev = h
        self.hidden = nn.ModuleList(layers)
        self.out = nn.Linear(prev, n_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = x
        for layer in self.hidden:
            h = self.act(layer(h))
        z = self.out(h)
        return z.squeeze(-1) if z.shape[-1] == 1 else z

model = SinMLP(1, HIDDEN_SIZES, n_out=1, activation=ACTIVATION).to(device)
crit = nn.MSELoss()
opt = torch.optim.AdamW(model.parameters(), lr=LR)

loss_hist = []
model.train()
for epoch in range(N_EPOCHS):
    opt.zero_grad()
    y_hat = model(x)
    loss = crit(y_hat, y)
    loss.backward()
    opt.step()
    if epoch % 100 == 0 or epoch == N_EPOCHS - 1:
        loss_hist.append(float(loss))

print("final MSE", round(loss_hist[-1], 4))


In [ ]:
model.eval()
with torch.no_grad():
    y_hat = model(x).cpu().numpy()

plt.figure(figsize=(8, 4))
plt.scatter(x_np, y_np, s=12, alpha=0.45, label="noisy data")
plt.plot(x_np, y_true_np, "k--", lw=2, label=r"true $\sin(x)$")
plt.plot(x_np, y_hat, "r", lw=2, label="MLP fit")
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"MLP regression — hidden={HIDDEN_SIZES}, act={ACTIVATION.__name__}")
plt.legend()
plt.tight_layout()
plt.show()


### Exercise 1 — architecture knobs

Change **one** of: `HIDDEN_SIZES`, `ACTIVATION`, `N_EPOCHS`, `NOISE_STD`. Re-run the three cells above (model, train, plot).

**Write one sentence:** what changed in the plot or final MSE?

**Desired:** deeper/wider nets usually track $\sin x$ more closely on this toy set; too little capacity underfits; very high noise makes any fit look worse.


## Part 2 — Same net, binary labels

Keep $x$, but label $y_i \in \{0,1\}$ by whether $\sin(x_i) > 0$. One **logit** $z$ per point; $\hat p = \sigma(z)$.

$$
z = \text{(last linear layer)}, \qquad \hat p = \sigma(z), \qquad
\mathcal{L} = \text{BCEWithLogits}(z, y).
$$

Last layer width = **1**. **No** `Sigmoid` inside the net when using `BCEWithLogitsLoss` (same rule as Lab 1).


In [ ]:
y_bin = torch.from_numpy((y_true_np > 0).astype(np.float32))

bin_model = SinMLP(1, HIDDEN_SIZES, n_out=1, activation=ACTIVATION)
bin_crit = nn.BCEWithLogitsLoss()
bin_opt = torch.optim.AdamW(bin_model.parameters(), lr=LR)

bin_model.train()
for _ in range(N_EPOCHS):
    bin_opt.zero_grad()
    logits = bin_model(x)
    loss = bin_crit(logits, y_bin)
    loss.backward()
    bin_opt.step()

bin_model.eval()
with torch.no_grad():
    p_hat = torch.sigmoid(bin_model(x))
    acc = float(((p_hat >= 0.5).float() == y_bin).float().mean())
print("binary acc", round(acc, 3))
# Desired: acc > 0.9 on this easy rule


### Exercise 2 — numeric vs binary

In one sentence: what changed in the **last layer** and the **loss** when you switched from Part 1 to Part 2?

**Desired:** still one hidden stack, but training target is class 0/1 and the loss is BCE on logits, not MSE on real numbers.


## Part 3 — Three classes (Ch. 2 multi-class head)

Now $K=3$ classes. Logits $\mathbf{z} \in \mathbb{R}^{K}$; predicted class probabilities

$$
\hat{\mathbf{p}} = \mathrm{softmax}(\mathbf{z}), \qquad
\hat p_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}.
$$

Loss (one-hot label with class index $y \in \{0,\ldots,K-1\}$):

$$
\mathcal{L} = -\log \hat p_y = \texttt{cross\_entropy}(\mathbf{z}, y).
$$

We use **toy tabular** rows (three features) so you can see the head before text embeddings.


In [ ]:
# three blobs in R^2 (lifted to 3-D for a nontrivial input)
torch.manual_seed(422)
K = 3
n_per = 80
centers = torch.tensor([[ -2., -1., 0.], [2., 1., 0.], [0., 2., -1.]])
xs, ys = [], []
for k in range(K):
    xs.append(centers[k] + 0.35 * torch.randn(n_per, 3))
    ys.append(torch.full((n_per,), k))
X3 = torch.cat(xs)
y3 = torch.cat(ys).long()

clf = SinMLP(3, [16, 16], n_out=K, activation=nn.ReLU)
opt3 = torch.optim.AdamW(clf.parameters(), lr=5e-3)
clf.train()
for _ in range(400):
    opt3.zero_grad()
    logits = clf(X3)                     # (n, 3)
    loss = F.cross_entropy(logits, y3)   # softmax + NLL inside
    loss.backward()
    opt3.step()

clf.eval()
with torch.no_grad():
  probs = F.softmax(clf(X3), dim=-1)
  pred = probs.argmax(dim=-1)
  acc3 = float((pred == y3).float().mean())
print("3-class acc", round(acc3, 3), "logits shape", tuple(logits.shape))
# Desired: acc3 > 0.95


### Exercise 3 — three-class knobs

Change hidden width or depth in `SinMLP(3, [16, 16], n_out=K, ...)`. Does accuracy change?

**Optional:** print `probs[0]` — three nonnegative numbers summing to 1.


## Part 4 — Tokenization (textbook / [theLMbook](https://github.com/aburkov/theLMbook))

Ch. 2 turns raw text into **token ids** before any neural net. The official notebooks use simple word splitting and a vocabulary map `stoi` / `itos`.

Steps:

1. **Tokenize** → list of word strings
2. **Build vocab** → map word $\to$ id $0,\ldots,V-1$
3. **Encode** → integer tensor of shape $(T,)$ or batched $(B, T)$

This is the same interface later LMs use — only the tokenizer (words vs subwords) changes.


In [ ]:
# tiny corpus (insurance-flavored, not the wildfire table)
CORPUS = [
    "fire risk rises in dry hills",
    "the model predicts serious damage",
    "tokens embed discrete words",
    "language models score next tokens",
    "softmax turns logits into probabilities",
]

def tokenize(text: str) -> list[str]:
    """Word-level tokenizer (Ch. 2 style; see theLMbook notebooks)."""
    return re.findall(r"\b\w+\b", text.lower())

words: list[str] = []
for line in CORPUS:
    words.extend(tokenize(line))

vocab = sorted(set(words))
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
V = len(vocab)

ids = torch.tensor([stoi[w] for w in words], dtype=torch.long)
print("V =", V, "tokens =", len(ids), "first 12 ids:", ids[:12].tolist())
print("decode:", " ".join(itos[i] for i in ids[:8].tolist()))


In [ ]:
C = 16  # embedding dimension (book: each id -> R^C)
emb = nn.Embedding(V, C)
x_ids = ids.unsqueeze(0)          # (1, T) batch of one sequence
h = emb(x_ids)                    # (1, T, C)
print("ids", tuple(x_ids.shape), "emb", tuple(h.shape))
# Desired: ids (1, T)  emb (1, T, 16)


### Exercise 4 — vocabulary

Add one new sentence to `CORPUS` with a word that never appeared before. Re-run tokenization.

**Desired:** $V$ increases by the number of **new** word types; OOV handling is a design choice (here: rebuild vocab).


## Part 5 — Next-token loss and **perplexity**

A language model assigns a distribution over the **next** token. For one position with logits $\mathbf{z} \in \mathbb{R}^{V}$ and true next id $y$:

$$
\mathcal{L}_{\text{NLL}} = -\log \hat p_y, \qquad \hat{\mathbf{p}} = \mathrm{softmax}(\mathbf{z}).
$$

**Perplexity** (book): average surprise per token, in "effective vocabulary size" units:

$$
\mathrm{PPL} = \exp(\overline{\mathcal{L}_{\text{NLL}}).
$$

Lower PPL on the **same** held-out text means the model assigns more mass to what actually occurred.


In [ ]:
class TinyNextToken(nn.Module):
    def __init__(self, vocab: int, n_emb: int = 32):
        super().__init__()
        self.emb = nn.Embedding(vocab, n_emb)
        self.head = nn.Linear(n_emb, vocab)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        # idx: (B, T) -> logits (B, T, V)
        return self.head(self.emb(idx))

lm = TinyNextToken(V, n_emb=32)
opt_lm = torch.optim.AdamW(lm.parameters(), lr=3e-3)

# one long sequence; predict token t+1 from prefix ending at t
seq = ids.unsqueeze(0)              # (1, T)
x_in = seq[:, :-1]
y_next = seq[:, 1:]

start = end = None
lm.train()
for step in range(300):
    opt_lm.zero_grad()
    logits = lm(x_in)
    loss = F.cross_entropy(logits.reshape(-1, V), y_next.reshape(-1))
    if step == 0:
        start = float(loss)
    loss.backward()
    opt_lm.step()
    end = float(loss)

ppl = math.exp(end)
print("start NLL", round(start, 3), "end NLL", round(end, 3), "perplexity", round(ppl, 2))
# Desired: end < start; PPL < V (often much smaller after training on this tiny corpus)


**Read the numbers:** if PPL $\approx V$, the model is near uniform guessing. If PPL $\approx 1$, it is very confident and usually right on this training snippet.

**Book contrast:** a count **bigram** model estimates $\hat P(w_t \mid w_{t-1})$ from table counts. Our `TinyNextToken` learns those probabilities with embeddings + a linear head.

### Exercise 5 — perplexity

Train with `n_emb=8` vs `n_emb=64` or fewer steps. Which run has **lower** PPL? One sentence tying PPL to "how surprised" the model is.


In [ ]:
# Autoregressive sampling: append each new token before the next forward pass
lm.eval()
ctx = torch.tensor([[stoi["fire"]]], dtype=torch.long)  # start word
generated = []
with torch.no_grad():
    for _ in range(8):
        logits = lm(ctx)[:, -1, :]           # last position only
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
        generated.append(itos[int(nxt)])
        ctx = torch.cat([ctx, nxt], dim=1)
print("seed: fire ->", " ".join(generated))
# Desired: short word string (may be nonsense on this tiny corpus)


## Before you leave

- [ ] Draw one hidden layer and label $\mathbf{W}^{(1)}, \mathbf{b}^{(1)}, \sigma$, and the output head.
- [ ] Part 1: changed depth/width/activation and read the $\sin x$ plot.
- [ ] Parts 2–3: binary (one logit) vs three-class (softmax / `cross_entropy`).
- [ ] Part 4–5: `stoi` / `itos`, `nn.Embedding` shapes, PPL $= \exp(\overline{\text{NLL}})$.
- [ ] [Quiz 2 bank](https://roverhol.github.io/data422/quizzes/quiz-02-prep.html)
